[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/lstm_text_generation_exercise.ipynb)

# Text generation: writing like Shakespeare

In this exercise you will train a recurrent neural network that writes text **one character at a time**. Like the pixel model of `autoregressive_generator.ipynb`, it is an **autoregressive** model: given the characters written so far, it predicts a probability distribution for the next one,

$$p(c_t \mid c_1, \dots, c_{t-1}),$$

we sample a character from it, append it to the text, and repeat.

### Dataset

We use **Tiny Shakespeare**, a single 1 MB text file with about 40 000 lines from Shakespeare's plays, put together by Andrej Karpathy for his [char-rnn](https://github.com/karpathy/char-rnn) project. It is a good first dataset for text generation because:

* it is one plain text file, with no cleaning to do;
* it has only a few dozen distinct characters, so we need no tokenizer and no word vocabulary;
* a small LSTM learns it in a few minutes, and you can easily tell whether the output "looks like" Shakespeare (character names, line breaks, old English words).

### How this notebook works

* Only the download of the dataset is given. Everything else is for you to write: cells with `...` or `# TODO` are for you to complete.
* Cells marked **Predict first** ask you to write what you *expect* **before** running the code.
* Cells marked **Explain** must be answered in your own words, referring to the **results you obtained**.
* `check(...)` cells test your work. If one fails, fix the cells above it before moving on.
* `lstm.ipynb` shows how to use `nn.Embedding`, `nn.LSTM`, `Dataset` and `DataLoader`. Use a GPU if you can (in Colab: *Runtime → Change runtime type*).

In [ ]:
import os
import urllib.request

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using", device)

torch.manual_seed(0)


def check(condition, message):
    if not condition:
        raise AssertionError("❌ " + message)
    print("✅", message)

## 1. Dataset

### 1.1 Download and load

The file is downloaded only once. `text` is a single Python string with the whole dataset.

In [ ]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
FILE = "tinyshakespeare.txt"
if not os.path.exists(FILE):
    urllib.request.urlretrieve(URL, FILE)

with open(FILE, encoding="utf-8") as f:
    text = f.read()

print(f"{len(text):,} characters")
print(text[:500])

### 1.2 Explore

Find the distinct characters of `text`, sorted, and print how many there are and what they are.

In [ ]:
# TODO: chars = sorted list of the distinct characters of text
chars = ...


In [ ]:
check(len(chars) == 65, "the text has 65 distinct characters")

**Explain:** Which kinds of characters appear? Are uppercase and lowercase letters the same character for the model? Why does the newline `\n` matter for the text we want to generate?

**Your answer:**



## 2. From characters to numbers

The network works with integers, not characters. Build:

* `char2idx`: a dictionary from each character to its index in `chars`;
* `idx2char`: the inverse mapping (a list or a dictionary);
* `encode(s)`: a string → a list of integers;
* `decode(ids)`: a list of integers → a string.

Then turn the whole text into a 1-D `torch.long` tensor `data`, and split it **by position**: the first 90% for training (`train_data`) and the last 10% for validation (`valid_data`).

In [ ]:
# TODO: char2idx, idx2char, vocab_size, encode, decode, data, train_data, valid_data


In [ ]:
check(vocab_size == 65, "vocab_size is 65")
check(decode(encode("To be, or not to be")) == "To be, or not to be", "decode(encode(s)) gives back s")
check(data.dtype == torch.long and data.shape == (len(text),), "data is a long tensor with one entry per character")
check(len(train_data) + len(valid_data) == len(data), "train and valid together have all the characters")

**Explain:** Why do we split the text by position instead of picking random characters (or random lines) for validation?

**Your answer:**



## 3. Training examples

The model learns to predict **every** next character of a sequence at once. From a chunk of `seq_len + 1` characters we take:

* the input `x`: the first `seq_len` characters;
* the target `y`: the last `seq_len` characters, i.e. the input **shifted by one**.

For example, with `seq_len = 4` and the chunk `"hello"`: `x = "hell"` and `y = "ello"`, so the model must predict `e` after `h`, `l` after `he`, and so on.

Write a `CharDataset(Dataset)` that receives a 1-D tensor and `seq_len` and returns non-overlapping chunks (`__getitem__(i)` starts at `i * seq_len`). Then create `train_loader` (shuffled) and `valid_loader`. Suggested values: `seq_len = 100`, `batch_size = 64`.

In [ ]:
seq_len = 100
batch_size = 64

# TODO: CharDataset, train_loader, valid_loader


In [ ]:
x, y = next(iter(train_loader))
check(x.shape == (batch_size, seq_len) and y.shape == (batch_size, seq_len), "a batch has shape (batch_size, seq_len)")
check(bool((x[:, 1:] == y[:, :-1]).all()), "the target is the input shifted by one character")
print(repr(decode(x[0][:60].tolist())))
print(repr(decode(y[0][:60].tolist())))

## 4. Model

Write `CharLSTM(nn.Module)` with three layers:

1. [`nn.Embedding`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)`(vocab_size, embedding_dim)`;
2. [`nn.LSTM`](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)`(embedding_dim, hidden_size, num_layers, batch_first=True)`;
3. `nn.Linear(hidden_size, vocab_size)`, applied to **every** time step.

`forward(x, state=None)` receives a batch of shape `(batch, time)` and returns the logits, of shape `(batch, time, vocab_size)`, **and** the new LSTM state `(h, c)`. Returning the state lets us continue a sequence one character at a time when generating. Suggested values: `embedding_dim = 64`, `hidden_size = 256`, `num_layers = 2`.

In [ ]:
# TODO: class CharLSTM(nn.Module)

model = ...


In [ ]:
logits, (h, c) = model(torch.zeros(2, 10, dtype=torch.long, device=device))
check(logits.shape == (2, 10, vocab_size), "logits have shape (batch, time, vocab_size)")

## 5. Generating text

Write `generate(model, start, length, temperature=1.0)`:

1. encode `start` and feed it to the model to get the logits and the state;
2. take the logits of the **last** time step, divide them by `temperature`, and turn them into probabilities with `softmax`;
3. sample the next character with [`torch.multinomial`](https://pytorch.org/docs/stable/generated/torch.multinomial.html);
4. feed **only that character** and the state back to the model, and repeat until you have `length` new characters.

Put the model in evaluation mode and use `torch.no_grad()`. Return `start` followed by the generated text.

In [ ]:
# TODO: def generate(model, start, length, temperature=1.0)


In [ ]:
sample = generate(model, "ROMEO:", 50)
check(sample.startswith("ROMEO:") and len(sample) == 56, "generate returns the start plus 50 new characters")

**Predict first:** What will the model write *before* training? Write your guess, then generate 300 characters starting with `"ROMEO:"`.

**Your answer:**



In [ ]:
# TODO: generate 300 characters with the untrained model


## 6. Training

[`nn.CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) expects logits of shape `(N, vocab_size)` and targets of shape `(N,)`, so flatten the batch and time dimensions (`logits.reshape(-1, vocab_size)`, `y.reshape(-1)`).

**Predict first:** A model that gives the same probability to the 65 characters has a cross-entropy of $\ln 65$. How much is that? Do you expect the trained model to be far below it?

**Your answer:**



Train the model with `Adam` (learning rate `2e-3`) for about 10 epochs. After each epoch:

* compute the average training loss and the validation loss (without gradients);
* print both, and a 200-character sample starting with `"ROMEO:"`.

Keep the losses in the lists `train_losses` and `valid_losses`, and plot them at the end.

In [ ]:
epochs = 10
train_losses, valid_losses = [], []

# TODO: training loop


In [ ]:
# TODO: plot train_losses and valid_losses


In [ ]:
check(len(valid_losses) > 0 and valid_losses[-1] < 2.0, "validation loss below 2.0")

**Explain:** How did the samples change along the epochs? What did the model learn first (spaces, words, names, line breaks, ...)? Is there any sign of overfitting in your plot?

**Your answer:**



## 7. Temperature

Generate 300 characters with `temperature` 0.2, 0.5, 1.0 and 1.5, always starting with `"ROMEO:"`.

In [ ]:
# TODO: one sample for each temperature


**Explain:** What does dividing the logits by the temperature do to the probabilities? Describe the text you got with a low and with a high temperature. Which value would you choose, and why?

**Your answer:**



## 8. Going further (optional)

* Train on another text. [Project Gutenberg](https://www.gutenberg.org/) has free books in plain text, for example *Os Lusíadas* in Portuguese. Remember that the vocabulary changes (accented letters).
* Replace the LSTM with an `nn.GRU`, or change `hidden_size` and `num_layers`. How do the number of parameters and the validation loss change?
* Use overlapping chunks (`__getitem__(i)` starting at `i`): you get many more training examples from the same text.

### AI usage declaration

If you used an AI tool (ChatGPT, Copilot, Gemini, Claude, ...) in this exercise, say **which one**, **for which sections**, and **what you asked it**. You may be asked to explain or change any cell of this notebook in class.

**Your answer:**

